# 📊 Sampling Techniques & Measures of Central Tendency

---

### 🎯 Why This Session Matters
In real-world Data Science, you almost never have access to every single record in an entire population—whether that is every customer in a global retail chain, every widget produced in a factory, or every citizen in a country. **Sampling** lets us take a smart, manageable slice of data to uncover truths about the whole population without wasting time or compute. **Measures of Central Tendency** then distill that data into a single representative "center" value.

### 🗺️ Complete Topic Roadmap
Every single topic, subtopic, comparison, and Python example from the lecture notes is covered below:
1. **Key Terminologies Glossary** — Plain-English definitions and memory hooks for all core terms.
2. **Introduction to Sampling Techniques** — What sampling is and why sample representativeness matters.
3. **Random Sampling** — Equal-probability selection, hat-draw analogy, and Python implementation (`df.sample()`).
4. **Stratified Sampling** — Subgroup (strata) division, proportional/equal representation, and Python implementation (`groupby().apply()`).
5. **Cluster Sampling** — Group-level selection (all members of chosen clusters) and Python implementation on Manufacturing Data (`BatchNumber`).
6. **Cluster Sampling vs. Stratified Sampling** — Deep-dive side-by-side comparison, workflows, key points, and when to choose which.
7. **Systematic Sampling** — Fixed-interval ($k$-th member) selection, sampling interval formula ($k = N/n$), and Python slicing (`df.iloc[::10, :]`).
8. **Sampling Bias & Errors** — How samples fail (Selection Bias, Non-Response Bias, Measurement Errors) and distort conclusions.
9. **Measures of Central Tendency: Mean, Median, and Mode** — Formulas, memory tricks, outlier sensitivity, and Python calculations (`.mean()`, `.median()`, `.mode()[0]`).
10. **End-to-End Python Lab: Comparing Sample Means vs. Actual Population Mean** — Full multi-store customer satisfaction & income simulation comparing all 4 sampling methods against ground truth.
11. **Master Revision Cheatsheet** — Scannable one-page quick reference for interviews and exams.

## 📖 1. Key Terminologies Glossary

Before diving into code and math, let's translate the statistical jargon into everyday language with quick memory hooks:

| Term | Plain-English Meaning | Memory Hook |
| :--- | :--- | :--- |
| **Population ($N$)** | The *entire* group you want to draw conclusions about (e.g., all 1,000 customers or every resident in a city). | **N** = **N**ationwide / **N**on-stop (everyone!). |
| **Sample ($n$)** | A smaller subset selected from the population to analyze and make inferences from. | Like tasting one spoonful of soup to judge the whole pot. |
| **Statistical Inference** | Using patterns found in a sample ($n$) to make educated conclusions about the whole population ($N$). | Connecting the dots from the *slice* to the *whole pie*. |
| **Stratum (pl. Strata)** | A non-overlapping, homogeneous subgroup within a population sharing a specific trait (e.g., age, grade, region). | Think of geological **strata** (distinct rock layers stacked neatly). |
| **Cluster** | A naturally occurring, pre-grouped mini-population (e.g., a school, a classroom, a manufacturing batch, a store). | A **cluster** of grapes—you grab a few whole bunches instead of picking single grapes. |
| **Sampling Interval ($k$)** | The fixed step size used in systematic sampling to pick every $k$-th item from an ordered list. | **k** = **k**eep skipping $k$ steps. |
| **Sampling Bias** | A systematic flaw in how a sample is collected that causes certain groups to be over- or under-represented. | A tilted scale that always adds extra weight to one side. |
| **Selection Bias** | Bias caused by choosing participants in a non-random way that excludes parts of the population. | Surveying only luxury mall shoppers to estimate a whole city's income. |
| **Non-Response Bias** | Bias caused when people who choose *not* to answer a survey differ meaningfully from those who *do*. | Only angry or super-happy customers bother filling out feedback forms. |
| **Measurement Error** | Inaccuracy between the true value and the recorded value due to faulty instruments, bad questions, or human typos. | A broken weighing scale reading 2 kg too heavy on every measurement. |
| **Mean ($\mu$ or $\bar{x}$)** | The arithmetic average: sum of all values divided by the count of values. | The "fair share" value if everyone pooled their money and split it equally. |
| **Median** | The exact middle value when all data points are sorted from smallest to largest. | The **median** strip right down the center of a highway. |
| **Mode** | The value that appears most frequently in a dataset. | **Mo**de = **Mo**st often. |

## 🎲 2. Introduction to Sampling & Simple Random Sampling

### What is Sampling?
**Sampling** is the process of selecting a subset of individuals or items ($n$) from a larger population ($N$) to make statistical inferences about the whole group. Because measuring an entire population is often too expensive, slow, or physically impossible (for instance, destructive quality testing in manufacturing), choosing the right sampling technique ensures our sample accurately mirrors the population.

---

### Technique 1: Simple Random Sampling
* **Definition:** Every member of the population has an **exact equal chance** of being selected, and each selection is completely independent of the others.
* **Everyday Analogy:** Writing every student's name on identical slips of paper, tossing them into a hat, shaking thoroughly, and drawing 10 names blindfolded.
* **Formula for Selection Probability:**
  $$P(\text{selecting any specific individual}) = \frac{n}{N}$$
  * **$n$:** Desired sample size.
  * **$N$:** Total population size.
  * **Memory Trick:** $\frac{\text{Needed}}{\text{Total Number}}$ — every person gets the exact same lottery odds!
* **When to Use It:** When you have a complete list of the entire population (a *sampling frame*) and the population is fairly uniform without critical tiny subgroups that might be missed by pure luck.
* **Common Trap:** Forgetting `random_state` in Python makes your sample change every time you re-run the notebook, making debugging and grading impossible.

In [3]:
import pandas as pd
import numpy as np

# Create a synthetic Population Survey Dataset (mirroring 'Population Survey Data.csv')
np.random.seed(42)
df_population = pd.DataFrame({
    'PersonID': range(1, 1001),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], size=1000, p=[0.40, 0.30, 0.20, 0.10]),
    'Income': np.random.normal(loc=55000, scale=15000, size=1000).round(2)
})

# Random Sampling on Population Data (n=100, fixed seed for reproducibility)
random_sample = df_population.sample(n=100, random_state=1)

print("Random Sample (First 5 rows):")
print(random_sample.head())
print(f"\nTotal rows sampled: {len(random_sample)} out of {len(df_population)}")

# Self-test assertion
assert len(random_sample) == 100, "Random sample should have exactly 100 rows!"
print("✅ Random sampling check passed!")

Random Sample (First 5 rows):
     PersonID Region    Income
507       508   East  62619.87
818       819  North  55311.91
452       453  South  53675.77
368       369  North  78917.58
242       243   West  54634.68

Total rows sampled: 100 out of 1000
✅ Random sampling check passed!


## 🍰 3. Stratified Sampling

### Concept & Intuition
* **Definition:** The population is first divided into distinct, non-overlapping homogeneous subgroups called **strata** based on a shared characteristic (e.g., Region, Age, Income Level, or University Major), and then a **random sample is drawn from each stratum**.
* **Everyday Analogy:** Imagine baking a 4-layer rainbow cake (red, yellow, green, blue). If you take a random scoop from the top, you might only get red frosting. Instead, slicing vertically through every layer guarantees your bite contains all 4 flavors!
* **Real-World Scenario:** Surveying students across different majors (Computer Science, Biology, History, Art) in a university. Even if Art majors make up only 5% of the university, stratified sampling guarantees their voices are captured rather than accidentally skipped by pure random chance.

### Formulas & Allocation Rules
1. **Equal Allocation (Fixed quota per stratum):**
   $$n_h = \text{constant (e.g., } 25 	\text{ per region)}$$
2. **Proportional Allocation (Mirrors population proportions):**
   $$n_h = n \times \frac{N_h}{N}$$
   * **$n_h$:** Sample size drawn from stratum $h$.
   * **$N_h$:** Total population size inside stratum $h$.
   * **$N$:** Total overall population size; **$n$:** Total target sample size.
   * **Memory Trick:** **S**tratified = **S**ubgroup **S**ampling (sample *inside every* slice).
* **Common Trap:** Using `min(len(x), 25)` inside `.apply()` is a great defensive coding habit—if a tiny stratum only has 18 records, asking for 25 without `replace=True` would throw a `ValueError`!

In [ ]:
# Stratified Sampling on Population Data by 'Region'
# Draws up to 25 random records from EACH Region stratum
stratified_sample = (
    df_population
    .groupby('Region')
    .apply(lambda x: x.sample(min(len(x), 25), random_state=42), include_groups=False)
    .reset_index(level=0)
)

# 1. Fixed Allocation (up to 25 records per Region)
# stratified_sample_fixed = (
#     df_population.sample(frac=1, random_state=42)
#     .groupby('Region', as_index=False)
#     .head(25)
# )
# 2. Proportional Allocation (10% from each Region)
# stratified_sample_prop = (
#     df_population.groupby('Region', group_keys=False)
#     .sample(frac=0.10, random_state=42 )
#     .reset_index(drop=True)
# )


print("Stratified Sample (First 5 rows):")
print(stratified_sample.head())

print("\nCount per Region in Stratified Sample:")
print(stratified_sample['Region'].value_counts())

# Self-test assertion
assert stratified_sample['Region'].nunique() == 4, "All 4 regions must be represented!"
print("✅ Stratified sampling check passed!")

Stratified Sample (First 5 rows):
    Region  PersonID    Income
972   East       973  45188.86
851   East       852  42304.65
91    East        92  43025.54
80    East        81  58043.85
331   East       332  58261.49

Count per Region in Stratified Sample:
Region
East     25
North    25
South    25
West     25
Name: count, dtype: int64
✅ Stratified sampling check passed!


## 🍇 4. Cluster Sampling

### Concept & Intuition
* **Definition:** The population is divided into pre-existing groups called **clusters** (usually based on geography, physical locations, or production batches). A **random subset of clusters is chosen**, and **100% of all members inside those chosen clusters** are sampled.
* **Everyday Analogy:** You are a quality inspector at a vineyard with 50 crates of grapes. Instead of opening all 50 crates to pull 2 grapes from each, you randomly pick 5 crates and inspect *every single grape* inside those 5 crates.
* **Real-World Scenarios:**
  * **Education:** Selecting 4 classrooms at random in a large school and surveying *every student* inside those 4 classrooms.
  * **Manufacturing Quality Control:** Selecting 5 production batches (`BatchNumber`) out of 30 batches and inspecting *every item* manufactured in those 5 batches.

### Formula & Mechanics
$$\text{Total Sample Size } (n) = \sum_{i \in S} M_i$$
* **$S$:** The set of randomly selected clusters (e.g., 5 chosen batches).
* **$M_i$:** The number of individual items inside selected cluster $i$.
* **Memory Trick:** **C**luster = **C**omplete **C**rate (pick a few groups, take *all* items inside them).
* **When to Use It:** When a complete list of every individual in the population does *not* exist, or when traveling to every location is too expensive, so grouping by geography or batch saves massive time and money.

In [12]:
# Create a synthetic Manufacturing Dataset (mirroring 'Manufacturing Data.csv')
np.random.seed(10)
batches = [f"BATCH_{i:02d}" for i in range(1, 21)]  # 20 distinct manufacturing batches
df_manufacturing = pd.DataFrame({
    'ItemID': range(1, 401),
    'BatchNumber': np.repeat(batches, 20),  # 20 items per batch
    'DefectScore': np.random.exponential(scale=1.5, size=400).round(3)
})

# Cluster Sampling on Manufacturing Data by 'BatchNumber'
clusters = df_manufacturing['BatchNumber'].unique()
selected_clusters = np.random.choice(clusters, size=5, replace=False)
cluster_sample = df_manufacturing[df_manufacturing['BatchNumber'].isin(selected_clusters)]

print(f"Selected 5 Clusters: {selected_clusters}")
print("\nCluster Sample (First 5 rows):")
print(cluster_sample.head())
print(f"\nTotal items inspected: {len(cluster_sample)} (All 20 items from each of the 5 chosen batches)")

# Self-test assertion
assert cluster_sample['BatchNumber'].nunique() == 5, "Exactly 5 clusters should be selected!"
assert len(cluster_sample) == 100, "5 batches x 20 items/batch = 100 items!"
print("✅ Cluster sampling check passed!")

Selected 5 Clusters: ['BATCH_14' 'BATCH_15' 'BATCH_12' 'BATCH_05' 'BATCH_06']

Cluster Sample (First 5 rows):
    ItemID BatchNumber  DefectScore
80      81    BATCH_05        1.395
81      82    BATCH_05        1.081
82      83    BATCH_05        1.366
83      84    BATCH_05        0.456
84      85    BATCH_05        0.537

Total items inspected: 100 (All 20 items from each of the 5 chosen batches)
✅ Cluster sampling check passed!


## ⚖️ 5. Cluster Sampling vs. Stratified Sampling (Crucial Comparison)

Students frequently confuse **Stratified** and **Cluster** sampling because both divide the population into groups first. The difference lies in **how you sample from those groups**:

```
STRATIFIED SAMPLING (Sample SOME from ALL groups)
Population ──▶ [Stratum A] ──▶ Pick a few randomly ──┐
           ──▶ [Stratum B] ──▶ Pick a few randomly ──┼──▶ Final Sample (All strata represented)
           ──▶ [Stratum C] ──▶ Pick a few randomly ──┘

CLUSTER SAMPLING (Sample ALL from SOME groups)
Population ──▶ [Cluster 1] ──▶ (Skipped)
           ──▶ [Cluster 2] ──▶ Randomly Chosen! ──▶ Take 100% of members ──┐
           ──▶ [Cluster 3] ──▶ (Skipped)                                   ├──▶ Final Sample
           ──▶ [Cluster 4] ──▶ Randomly Chosen! ──▶ Take 100% of members ──┘
```

### Detailed Side-by-Side Breakdown

| Feature | Stratified Sampling | Cluster Sampling |
| :--- | :--- | :--- |
| **Workflow** | 1. Define population<br>2. Divide into homogeneous strata<br>3. Randomly select members **from every stratum**<br>4. Combine into sample | 1. Define population<br>2. Divide into naturally occurring clusters<br>3. Randomly select **a few clusters**<br>4. Collect data from **all members** of chosen clusters |
| **Core Focus** | Focuses on **individual members within each stratum**. | Focuses on **entire clusters**, not individual members. |
| **Group Composition** | Groups are **homogeneous** inside (e.g., all 9th graders in one group, all 10th graders in another). | Groups are **mini-populations** (heterogeneous inside, similar to each other, e.g., different schools in a city). |
| **City Student Example** | Divide students by grade level (9th, 10th, 11th, 12th) and randomly pick a proportional number of students from **every grade**. | Randomly pick 5 schools out of 50 in the city and survey **every single student** in those 5 schools. |
| **Primary Advantage** | **Higher precision & accuracy:** Guarantees every subgroup is represented, reflecting full population diversity. | **Cost-effective & practical:** Works when a full individual list isn't available and saves travel/logistics costs for large, spread-out populations. |

> **Golden Rule for Interviews:**  
> * **Stratified** = **All** groups, **Some** members.  
> * **Cluster** = **Some** groups, **All** members.

## 📏 6. Systematic Sampling

### Concept & Intuition
* **Definition:** Members of the population are arranged in an order (or arrive sequentially), and **every $k$-th (or $n$-th) member** is selected after a random starting point.
* **Everyday Analogy:** Standing outside a retail store exit and handing a 1-minute survey tablet to **every 10th shopper** who walks through the door.
* **Real-World Scenario:** Quality control on an automated factory assembly line—pulling **every 10th bottle** coming off the conveyor belt to verify fill volume and cap seal.

### Formula & Memory Trick
$$k = \left\lfloor \frac{N}{n} \right\rfloor$$
* **$k$:** Sampling interval (step size).
* **$N$:** Estimated total population size.
* **$n$:** Desired sample size.
* **Step-by-Step Workflow:**
  1. Define the population ($N$).
  2. Decide on the desired sample size ($n$).
  3. Calculate the sampling interval $k = N / n$ (e.g., $1000 / 100 = 10$).
  4. Select a random starting index between $0$ and $k-1$, then select every $k$-th member (`df.iloc[start::k, :]`).
* **Common Trap (Hidden Periodicity):** If the dataset has a repeating cycle that matches $k$ (e.g., every 10th item on a machine is welded by a faulty nozzle, or every 7th day in a daily log is Sunday), systematic sampling will produce a wildly biased sample!

In [16]:
# Systematic Sampling on Population Data (selecting every 10th row)
k = 10
systematic_sample = df_population.iloc[::k, :]

print("Systematic Sample (First 5 rows — notice PersonID steps by 10):")
print(systematic_sample.head())
print(f"\nTotal rows selected: {len(systematic_sample)} (1000 / 10 = 100)")

# Bonus: Systematic sampling with a random start between 0 and k-1
np.random.seed(42)
random_start = np.random.randint(0, k)
systematic_random_start = df_population.iloc[random_start::k, :]
print(f"With random start index {random_start}, first PersonID is: {systematic_random_start.iloc[0]['PersonID']}")

assert len(systematic_sample) == 100, "Systematic step of 10 on 1000 rows yields 100 rows!"
print("✅ Systematic sampling check passed!")

Systematic Sample (First 5 rows — notice PersonID steps by 10):
    PersonID Region    Income
0          1  North  57665.52
10        11  North  61444.27
20        21  South  69074.26
30        31  South  52196.93
40        41  North  47240.67

Total rows selected: 100 (1000 / 10 = 100)
With random start index 6, first PersonID is: 7
✅ Systematic sampling check passed!


## ⚠️ 7. Sampling Bias & Errors

Even the best statistical formulas cannot rescue a dataset poisoned by bad data collection. **Sampling bias** occurs when a sample does not accurately represent the target population, leading to skewed, misleading conclusions.

### The Three Major Sources of Error & Bias

1. **Selection Bias**
   * **What it is:** The method used to pick participants systematically excludes or favors certain segments of the population.
   * **Real-World Example:** Surveying residents *only* in a wealthy, high-income neighborhood to estimate the average income of an entire city. Your estimate will be drastically inflated because middle- and low-income neighborhoods had zero chance of selection.
2. **Non-Response Bias**
   * **What it is:** A large portion of selected individuals refuse or fail to respond, and those non-responders hold systematically different opinions or traits than those who do respond.
   * **Real-World Example:** Sending an optional email survey about internet speed. Customers experiencing terrible outages can't load the form (or frustrated customers flood it), meaning the responses fail to represent the silent majority.
3. **Measurement Errors**
   * **What it is:** The recorded data differs from the true value due to poorly worded survey questions, miscalibrated sensors, rounding errors, or manual data-entry mistakes.
   * **Real-World Example:** Asking *"Don't you agree our customer service is amazing?"* (a leading question) or logging income in thousands (`55`) for some rows and full dollars (`55000`) for others.

### Total Error Decomposition (Conceptual Formula)
$$\text{Total Survey Error} = \text{Sampling Error (random chance)} + \text{Non-Sampling Error (Bias + Measurement Error)}$$
* **Key Insight:** Increasing your sample size ($n$) reduces *random sampling error*, **but it does NOT fix Selection Bias or Measurement Error!** A million biased responses are still biased.

In [17]:
# Demonstration: How Selection Bias distorts the estimated Population Mean
true_mean_income = df_population['Income'].mean()

# Biased sample: Only surveying the highest-earning top 20% of the population
biased_sample = df_population[df_population['Income'] > df_population['Income'].quantile(0.80)]
biased_mean_income = biased_sample['Income'].mean()

print(f"True Population Mean Income   : ${true_mean_income:,.2f}")
print(f"Biased Sample Mean Income     : ${biased_mean_income:,.2f}")
print(f"Inflation caused by Bias      : +${biased_mean_income - true_mean_income:,.2f} (+{((biased_mean_income/true_mean_income)-1)*100:.1f}%)")

True Population Mean Income   : $56,483.44
Biased Sample Mean Income     : $77,420.77
Inflation caused by Bias      : +$20,937.34 (+37.1%)


## 🎯 8. Measures of Central Tendency: Mean, Median, and Mode

**Measures of central tendency** summarize an entire distribution of numbers using a single representative "center" value.

### 1. Mean (Arithmetic Average)
* **Definition:** The sum of all data points divided by the total number of data points.
* **Formula:**
  $$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i = \frac{x_1 + x_2 + \dots + x_n}{n}$$
  * **$x_i$:** Each individual data value.
  * **$n$:** Total number of observations.
  * **Memory Trick:** **Mean** is "mean" when extreme outliers show up—one billionaire walking into a coffee shop makes the *mean* net worth of everyone inside jump into the millions!
* **Real-World Example:** Calculating the average monthly income of residents in a city or average daily temperature.
* **When to Use / Trap:** Best for symmetric distributions without extreme outliers. Avoid as the sole metric on heavily skewed data (like household wealth or house prices).

---

### 2. Median (The Middle Value)
* **Definition:** The exact middle value when all data points are sorted in ascending (or descending) order.
* **Formula (for sorted data $x_{(1)} \le x_{(2)} \le \dots \le x_{(n)}$):**
  $$\text{Median} = \begin{cases} x_{\left(\frac{n+1}{2}\right)} & \text{if } n \text{ is odd} \\ \frac{x_{\left(\frac{n}{2}\right)} + x_{\left(\frac{n}{2} + 1\right)}}{2} & \text{if } n \text{ is even} \end{cases}$$
  * **Memory Trick:** Just like the concrete **median** strip divides a two-way highway right down the middle, 50% of values sit below it and 50% sit above it.
* **Real-World Example:** Median age of citizens in a country or median home price in a metro area.
* **When to Use:** Ideal for **skewed data or datasets with extreme outliers**, because moving the largest number from $100,000 to $100,000,000 doesn't budge the middle value at all!

---

### 3. Mode (Most Frequent Value)
* **Definition:** The value (or category) that occurs most frequently in the dataset.
* **Formula / Rule:**
  $$\text{Mode} = \arg\max_{x} \text{Frequency}(x)$$
  * **Memory Trick:** **Mo**de = **Mo**st often.
* **Real-World Example:** The most common shoe size sold in a footwear store (e.g., Size 9)—a store manager needs to stock Size 9 boxes, not "Size 8.63" (the mean)!
* **When to Use / Trap:** The **only** measure of central tendency that works on categorical/non-numeric data (like favorite color or store location). Note that a dataset can have no mode, one mode (unimodal), or multiple modes (bimodal/multimodal)—which is why Pandas `.mode()` returns a `Series` and we index `[0]` (`df['col'].mode()[0]`).

In [18]:
# Calculate Mean, Median, and Mode for Income on Population Data
mean_income = df_population['Income'].mean()
median_income = df_population['Income'].median()
mode_income = df_population['Income'].mode()[0]

print(f"Mean Income   : ${mean_income:,.2f}")
print(f"Median Income : ${median_income:,.2f}")
print(f"Mode Income   : ${mode_income:,.2f}")

# Quick demonstration of Outlier Resistance (Mean vs. Median)
salaries = pd.Series([45000, 48000, 50000, 52000, 55000])
salaries_with_ceo = pd.Series([45000, 48000, 50000, 52000, 5000000])  # $5M outlier!

print("\n--- Outlier Impact Test ---")
print(f"Normal Team     -> Mean: ${salaries.mean():,.2f} | Median: ${salaries.median():,.2f}")
print(f"With $5M Outlier-> Mean: ${salaries_with_ceo.mean():,.2f} | Median: ${salaries_with_ceo.median():,.2f}")

Mean Income   : $56,483.44
Median Income : $56,263.21
Mode Income   : $11,179.74

--- Outlier Impact Test ---
Normal Team     -> Mean: $50,000.00 | Median: $50,000.00
With $5M Outlier-> Mean: $1,039,000.00 | Median: $50,000.00


## 🧪 9. End-to-End Python Lab: Comparing Sample Means vs. Actual Mean

Let's bring everything together using the exact multi-store customer scenario from the session notes:

> **Business Scenario:** A retail company operates across 4 regions (`North`, `South`, `East`, `West`) and wants to estimate customer `Income` and `Satisfaction` across 1,000 customers without surveying everyone:
> * **Random Sampling:** Pick 100 customers at random from the entire database.
> * **Stratified Sampling:** Group by `StoreLocation` and randomly pick 25 customers from *each* location ($4 	imes 25 = 100$ customers).
> * **Cluster Sampling:** Randomly pick 2 `StoreLocation` clusters and survey *every* customer in those 2 stores.
> * **Systematic Sampling:** Pick every 10th customer (`df.iloc[::10, :]`, giving 100 customers).

Run the cell below to execute the exact benchmark and see how close each sampling technique gets to the **Actual Mean Income**!

In [21]:
import pandas as pd
import numpy as np

# Sample dataset (Exact setup from Session 03 Notes)
np.random.seed(42)
df_customers = pd.DataFrame({
    'CustomerID': range(1, 1001),
    'StoreLocation': np.random.choice(['North', 'South', 'East', 'West'], 1000),
    'Satisfaction': np.random.randint(1, 6, 1000),
    'Income': np.random.randint(20000, 80000, 1000)
})

# 1. Actual Population Mean Income
actual_mean_income = df_customers['Income'].mean()

# 2. Random Sampling
random_sample = df_customers.sample(n=100, random_state=42)
random_sample_mean_income = random_sample['Income'].mean()

# 3. Stratified Sampling (25 from each StoreLocation)
stratified_sample_fixed = (
    df_population.sample(frac=1, random_state=42)
    .groupby('Region', as_index=False)
    .head(25)
)
stratified_sample_mean_income = stratified_sample['Income'].mean()

# 4. Cluster Sampling (2 StoreLocations selected, all customers included)
clusters = df_customers['StoreLocation'].unique()
selected_clusters = np.random.choice(clusters, size=2, replace=False)
cluster_sample = df_customers[df_customers['StoreLocation'].isin(selected_clusters)]
cluster_sample_mean_income = cluster_sample['Income'].mean()

# 5. Systematic Sampling (Every 10th customer)
systematic_sample = df_customers.iloc[::10, :]
systematic_sample_mean_income = systematic_sample['Income'].mean()

# Summary Comparison Table
results = pd.DataFrame({
    'Sampling Technique': [
        'Actual Population (Ground Truth)',
        'Simple Random Sampling',
        'Stratified Sampling (by StoreLocation)',
        f'Cluster Sampling ({list(selected_clusters)})',
        'Systematic Sampling (Every 10th)'
    ],
    'Sample Size (n)': [
        len(df_customers),
        len(random_sample),
        len(stratified_sample),
        len(cluster_sample),
        len(systematic_sample)
    ],
    'Mean Income ($)': [
        actual_mean_income,
        random_sample_mean_income,
        stratified_sample_mean_income,
        cluster_sample_mean_income,
        systematic_sample_mean_income
    ]
})

results['Error vs. Actual ($)'] = results['Mean Income ($)'] - actual_mean_income
results['Abs Error (%)'] = (results['Error vs. Actual ($)'].abs() / actual_mean_income) * 100

print(results.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

                    Sampling Technique  Sample Size (n)  Mean Income ($)  Error vs. Actual ($)  Abs Error (%)
      Actual Population (Ground Truth)             1000        50,542.18                  0.00           0.00
                Simple Random Sampling              100        50,604.66                 62.48           0.12
Stratified Sampling (by StoreLocation)              100        53,891.54              3,349.36           6.63
  Cluster Sampling (['West', 'North'])              538        50,309.81               -232.37           0.46
      Systematic Sampling (Every 10th)              100        51,559.64              1,017.46           2.01


## 📋 10. End-of-Notebook Master Revision Cheatsheet

Keep this quick-reference table handy for rapid review before interviews, quizzes, or data analysis projects:

| Concept | Core Definition / Formula | Pandas / NumPy Syntax | Best Used When... | Watch Out For... |
| :--- | :--- | :--- | :--- | :--- |
| **Random Sampling** | Equal chance $P = n/N$ for every member. | `df.sample(n=100, random_state=1)` | You have a complete list and a relatively uniform population. | Small minority subgroups might be missed by chance. |
| **Stratified Sampling** | Divide into homogeneous strata; sample from **every** stratum ($n_h = n \cdot N_h/N$). | `df.groupby('Col').apply(lambda x: x.sample(min(len(x), 25, random_state=42), include_groups=False).reset_index(level=0)` | You must guarantee representation of all subgroups (e.g., regions, grades). | Requires knowing the stratum label for every record upfront. |
| **Cluster Sampling** | Divide into clusters; randomly select a few clusters and sample **100%** of their members. | `np.random.seed(42); chosen = np.random.choice(clusters, size=2, replace=False); df[df['Col'].isin(chosen)]` | Population is geographically spread out or no master individual list exists. | Higher sampling error if clusters differ drastically from one another. |
| **Systematic Sampling** | Pick every $k$-th item where $k = \lfloor N/n \rfloor$. | `df.iloc[::10, :]` | Sequential streams (production lines, store foot traffic). | Hidden periodicity/cycles in the data matching interval $k$. |
| **Selection Bias** | Sample systematically excludes part of the target group. | *Fix via proper sampling design* | — | Cannot be fixed merely by increasing sample size $n$. |
| **Non-Response Bias** | Non-responders differ from responders. | *Fix via follow-ups & weighting* | — | Voluntary surveys skew toward extreme opinions. |
| **Measurement Error** | Recorded values differ from true values. | *Fix via data validation & calibration* | — | Leading survey questions or unit mismatches (`55` vs `55000`). |
| **Mean ($\bar{x}$)** | $\frac{1}{n}\sum_{i=1}^n x_i$ (Arithmetic average) | `df['Income'].mean()` | Symmetric distributions without extreme outliers. | Heavily pulled toward extreme outliers/skew. |
| **Median** | Middle sorted value ($50\text{th}$ percentile). | `df['Income'].median()` | Skewed data (incomes, home prices) or data with outliers. | Ignores exact magnitudes of extreme values. |
| **Mode** | Most frequently occurring value. | `df['Income'].mode()[0]` | Categorical data or discrete sizing/inventory decisions. | Datasets can be multimodal (always index `[0]` or inspect all modes). |